# Exploración del catálogo Data360 — notebook autocontenido

Este notebook reproduce las **6 visualizaciones de exploración** del Paso 1 del TP Final consultando la API de Data360 directamente. No lee CSV ni depende de scripts del repositorio: se puede compartir y ejecutar de punta a punta.

**Qué hace**

1. Descarga los metadatos de los ~9.859 indicadores (paginado de a 1.000).
2. Descarga la jerarquía país → región.
3. Construye en memoria las tablas de indicadores, bases, cobertura regional y frescura.
4. Genera las 6 figuras con Altair / Vega-Lite.

**Requisitos:** conexión a internet y Python con `pandas`, `requests`, `altair` y `squarify`. La primera celda instala lo que falte.

**Tiempo estimado:** 1–2 minutos. La verificación de frescura hace ~340 consultas al endpoint de datos.

**Fecha de referencia:** se toma la fecha de ejecución (`date.today()`) para calcular la frescura.

**Enfoque:** el notebook prioriza las figuras y la lectura del catálogo; las tablas intermedias quedan en memoria y solo se imprimen los totales que sostienen la narrativa.


In [1]:
from __future__ import annotations

import importlib.util
import statistics
import subprocess
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date, datetime

for _package in ("pandas", "requests", "altair", "squarify"):
    if importlib.util.find_spec(_package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _package])

import altair as alt
import pandas as pd
import requests
import squarify

pd.set_option("display.max_columns", 50)
alt.data_transformers.enable("default")

# Paleta global: tonos muted compartidos por todas las figuras.
# La barra de periodicidad define los colores base; el resto de los gráficos
# reutiliza esos mismos valores para mantener una identidad visual consistente.
PERIODICITY_COLORS = {
    "Anual": "#5B7C99",
    "Desconocida": "#B7B7B7",
    "Trienal": "#D99A6C",
    "Quinquenal": "#7FA88B",
    "Mensual": "#C48B9F",
    "Trimestral": "#93B7C9",
    "Bienal": "#C97A6D",
    "Cada dos años": "#D9C58F",
    "Ad hoc (irregular)": "#9C8573",
    "Otra": "#9B8BB4",
}
CATEGORICAL_PALETTE = list(PERIODICITY_COLORS.values())

THEME_COLORS = {
    "Prosperity": PERIODICITY_COLORS["Anual"],
    "Sin clasificar": PERIODICITY_COLORS["Desconocida"],
    "People": PERIODICITY_COLORS["Trienal"],
    "Infrastructure": PERIODICITY_COLORS["Quinquenal"],
    "Planet": PERIODICITY_COLORS["Trimestral"],
    "Digital": PERIODICITY_COLORS["Otra"],
}

COVERAGE_COLORS = ["#E8EEF2", "#C5D3DD", "#93B7C9", "#5B7C99", "#3A556B"]

DUMBBELL_COLORS = {
    "Último dato real": PERIODICITY_COLORS["Anual"],
    "Fin nominal (metadata)": PERIODICITY_COLORS["Bienal"],
}

GROUP_COLORS = {
    "Shortlist temático": PERIODICITY_COLORS["Bienal"],
    "Resto del catálogo": PERIODICITY_COLORS["Desconocida"],
}
GROUP_TEXT_COLOR = "#8C4A3F"

META_API = "https://data360api.worldbank.org/data360/metadata"
DATA_API = "https://data360api.worldbank.org/data360/data"
HIERARCHY_API = "https://data360api.worldbank.org/data360/portal/v1/hierarchy"
DATASET_SEARCH_API = (
    "https://data360api.worldbank.org/data360/portal/v1/public_data360_search"
)

PAGE = 1000
MAX_WORKERS = 8
REFERENCE_DATE = date.today()
REFERENCE_YEAR = REFERENCE_DATE.year

SELECT = ",".join(
    [
        "tags",
        "series_description/idno",
        "series_description/name",
        "series_description/database_id",
        "series_description/database_name",
        "series_description/periodicity",
        "series_description/time_periods",
        "series_description/topics",
        "series_description/ref_country",
        "series_description/measurement_unit",
        "series_description/date_last_update",
        "series_description/date_released",
    ]
)
QUERY = (
    f"&$top={PAGE}&$filter=type eq 'indicator'&$select={SELECT}"
    "&$orderby=series_description%2Fdatabase_id%20asc,series_description%2Fidno%20asc"
)

TOPIC_VOCAB = {
    "WB Practice Groups": "L0",
    "Data360 Topic L1": "L1",
    "Data360 Topic L2": "L2",
}
NON_COUNTRY = {
    "AFE", "AFW", "EAS", "ECS", "LCN", "MEA", "NAC", "SAS", "SSF",
    "ARB", "WLD", "EUU", "EMU", "OED", "HIC", "LIC", "LMC", "MIC", "UMC",
    "INX", "IBD", "IDA", "IDB",
}
GRANULAR = {"Monthly", "Quarterly", "Weekly", "Daily"}

SESSION = requests.Session()
SESSION.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) world-bank-infoviz/0.1",
    }
)


def parse_period_end(value: str | None) -> date | None:
    if not value:
        return None
    value = value.strip()
    try:
        if "-Q" in value:
            year, quarter = value.split("-Q")
            month = int(quarter) * 3
            last_day = [31, 30, 30, 31][int(quarter) - 1]
            return date(int(year), month, last_day)
        parts = value.split("-")
        if len(parts) == 1:
            return date(int(parts[0]), 12, 31)
        if len(parts) == 2:
            year, month = int(parts[0]), int(parts[1])
            if month == 12:
                return date(year, 12, 31)
            first_next = date(year, month + 1, 1)
            return first_next.fromordinal(first_next.toordinal() - 1)
        return datetime.fromisoformat(value).date()
    except (ValueError, TypeError):
        return None


def normalize_periodicity(value: str | None) -> str:
    if not value:
        return "Unknown"
    lowered = value.lower()
    if "annual" in lowered or "year" in lowered or lowered == "ad hoc [adhoc]":
        return "Annual" if "annual" in lowered else value
    if "quarter" in lowered:
        return "Quarterly"
    if "month" in lowered:
        return "Monthly"
    if "week" in lowered:
        return "Weekly"
    if "daily" in lowered or "day" in lowered:
        return "Daily"
    if "triennial" in lowered:
        return "Triennial"
    if "biennial" in lowered or "two years" in lowered:
        return "Biennial"
    return value


def parse_period(value: str | None) -> date | None:
    value = str(value or "").strip()
    try:
        if "-Q" in value:
            year, quarter = value.split("-Q")
            return date(int(year), int(quarter) * 3, 1)
        parts = value.split("-")
        if len(parts) == 1:
            return date(int(parts[0]), 12, 31)
        if len(parts) == 2:
            return date(int(parts[0]), int(parts[1]), 1)
        return date.fromisoformat(value)
    except (ValueError, TypeError):
        return None


def fetch_metadata() -> list[dict]:
    records: list[dict] = []
    seen: set[tuple[str | None, str | None]] = set()
    skip = 0
    total = None
    while True:
        payload = None
        for attempt in range(5):
            try:
                response = SESSION.post(
                    META_API,
                    json={"query": f"{QUERY}&$skip={skip}"},
                    timeout=180,
                )
                response.raise_for_status()
                payload = response.json()
                break
            except (requests.RequestException, ValueError):
                if attempt == 4:
                    raise
                time.sleep(2 * (attempt + 1))
        if payload is None:
            break
        if total is None:
            total = payload.get("@odata.count")
        page = payload.get("value") or []
        if not page:
            break
        for document in page:
            series = document.get("series_description") or {}
            key = (series.get("database_id"), series.get("idno"))
            if key in seen:
                continue
            seen.add(key)
            records.append(
                {
                    "idno": series.get("idno"),
                    "name": series.get("name"),
                    "database_id": series.get("database_id"),
                    "database_name": series.get("database_name"),
                    "periodicity": series.get("periodicity"),
                    "time_periods": series.get("time_periods"),
                    "topics": series.get("topics"),
                    "ref_country": series.get("ref_country"),
                    "measurement_unit": series.get("measurement_unit"),
                    "date_last_update": series.get("date_last_update"),
                    "date_released": series.get("date_released"),
                    "tags": document.get("tags"),
                }
            )
        if len(page) < PAGE:
            break
        skip += PAGE
    return records


def fetch_hierarchy() -> list[dict]:
    response = SESSION.get(HIERARCHY_API, params={"type": "country"}, timeout=60)
    response.raise_for_status()
    return response.json()["data"]["HCL_WB_REGIONS"]


def fetch_dataset_names() -> dict[str, str]:
    names: dict[str, str] = {}
    skip = 0
    page_size = 200
    while True:
        response = SESSION.post(
            DATASET_SEARCH_API,
            json={
                "site": "data360",
                "query_string": "",
                "types": ["dataset"],
                "data_classification": ["public"],
                "skip": skip,
                "items_per_page": page_size,
            },
            timeout=60,
        )
        response.raise_for_status()
        payload = response.json()
        results = payload.get("results") or []
        for item in results:
            idno = item.get("idno")
            name = item.get("name")
            if idno and name:
                names[idno] = name
        skip += len(results)
        if not results or skip >= payload.get("count", 0):
            break
    return names


def get_data(params: dict) -> dict:
    for attempt in range(4):
        try:
            response = SESSION.get(DATA_API, params=params, timeout=60)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError):
            if attempt == 3:
                return {}
            time.sleep(0.5 * (attempt + 1))
    return {}


def latest_for(
    database_id: str, indicator_id: str, country: str
) -> tuple[date | None, str | None, int]:
    payload = get_data(
        {
            "DATABASE_ID": database_id,
            "INDICATOR": indicator_id,
            "REF_AREA": country,
            "timePeriodFrom": 2010,
            "timePeriodTo": REFERENCE_YEAR,
        }
    )
    rows = payload.get("value") or []
    best = None
    best_raw = None
    for row in rows:
        parsed = parse_period(row.get("TIME_PERIOD"))
        if parsed and (best is None or parsed > best):
            best, best_raw = parsed, row.get("TIME_PERIOD")
    return best, best_raw, len(rows)


## 1. Descarga de metadatos

Se usa `POST /data360/metadata` con sintaxis OData. La API devuelve como máximo 1.000 documentos por página; se pagina con `$skip` y se ordena por `database_id, idno` para evitar duplicados. El total viene en `@odata.count`.


In [2]:
started = time.time()
records = fetch_metadata()
elapsed = time.time() - started
databases_found = len(
    {record["database_id"] for record in records if record.get("database_id")}
)
print(f"Metadatos: {len(records):,} indicadores · {databases_found:,} bases · {elapsed:.1f} s")


Metadatos: 9,859 indicadores · 171 bases · 38.5 s


## 2. Indicadores y bases

De cada documento se extraen período, periodicidad, cobertura declarada (`ref_country`), temas L0/L1/L2 y región. La cobertura es **nominal**: cantidad de economías declaradas en metadata, no densidad real de observaciones.


In [3]:
def build_indicators(records: list[dict]) -> pd.DataFrame:
    rows = []
    for record in records:
        periods = record.get("time_periods") or []
        starts = [period.get("start") for period in periods if period.get("start")]
        ends = [period.get("end") for period in periods if period.get("end")]
        start = min(starts) if starts else None
        end = max(ends) if ends else None
        end_date = parse_period_end(end)

        topic_names: dict[str, list[str]] = defaultdict(list)
        for topic in record.get("topics") or []:
            level = TOPIC_VOCAB.get(topic.get("vocabulary"), "other")
            name = topic.get("name") or topic.get("id") or ""
            if name:
                topic_names[level].append(name)

        regions = {
            tag.get("tag")
            for tag in record.get("tags") or []
            if tag.get("tag_group") == "overview-region-profile-region"
        }

        rows.append(
            {
                "database_id": record["database_id"] or "UNKNOWN",
                "idno": record.get("idno"),
                "name": record.get("name"),
                "periodicity": normalize_periodicity(record.get("periodicity")),
                "period_start": start,
                "period_end": end,
                "period_end_date": end_date.isoformat() if end_date else "",
                "coverage_economies": len(record.get("ref_country") or []),
                "regions": "|".join(sorted(regions)),
                "topic_l0": "|".join(sorted(set(topic_names.get("L0", [])))),
                "topic_l1": "|".join(sorted(set(topic_names.get("L1", [])))),
                "topic_l2": "|".join(sorted(set(topic_names.get("L2", [])))),
                "measurement_unit": record.get("measurement_unit"),
            }
        )
    return pd.DataFrame(rows)


indicators = build_indicators(records)
annual_share = (indicators["periodicity"] == "Annual").mean()
median_coverage = indicators["coverage_economies"].median()
print(
    f"Indicadores: {len(indicators):,} · anuales: {annual_share:.1%} · "
    f"cobertura mediana: {median_coverage:.0f} economías"
)


Indicadores: 9,859 · anuales: 95.3% · cobertura mediana: 141 economías


In [4]:
def build_databases(records: list[dict]) -> pd.DataFrame:
    db_indicators: Counter[str] = Counter()
    db_names: dict[str, str] = {}
    db_periodicity: dict[str, Counter[str]] = defaultdict(Counter)
    db_coverage: dict[str, list[int]] = defaultdict(list)
    db_topics: Counter[tuple[str, str]] = Counter()
    db_regions: dict[str, set[str]] = defaultdict(set)
    db_span_start: dict[str, date] = {}
    db_span_end: dict[str, date] = {}

    for record in records:
        database_id = record["database_id"] or "UNKNOWN"
        db_indicators[database_id] += 1
        if record.get("database_name") and record["database_name"] != database_id:
            db_names[database_id] = record["database_name"]
        db_periodicity[database_id][
            normalize_periodicity(record.get("periodicity"))
        ] += 1
        db_coverage[database_id].append(len(record.get("ref_country") or []))

        periods = record.get("time_periods") or []
        starts = [
            parse_period_end(period.get("start"))
            for period in periods
            if period.get("start")
        ]
        ends = [
            parse_period_end(period.get("end"))
            for period in periods
            if period.get("end")
        ]
        starts = [value for value in starts if value]
        ends = [value for value in ends if value]
        if starts:
            start = min(starts)
            if database_id not in db_span_start or start < db_span_start[database_id]:
                db_span_start[database_id] = start
        if ends:
            end = max(ends)
            if database_id not in db_span_end or end > db_span_end[database_id]:
                db_span_end[database_id] = end

        for topic in record.get("topics") or []:
            if TOPIC_VOCAB.get(topic.get("vocabulary")) == "L1":
                name = topic.get("name") or topic.get("id") or ""
                if name:
                    db_topics[(database_id, name)] += 1

        db_regions[database_id] |= {
            tag.get("tag")
            for tag in record.get("tags") or []
            if tag.get("tag_group") == "overview-region-profile-region"
        }

    rows = []
    for database_id, count in sorted(
        db_indicators.items(), key=lambda item: (-item[1], item[0])
    ):
        coverage = db_coverage[database_id]
        top_l1 = sorted(
            [
                (name, value)
                for (db, name), value in db_topics.items()
                if db == database_id
            ],
            key=lambda item: (-item[1], item[0]),
        )[:3]
        rows.append(
            {
                "database_id": database_id,
                "database_name": db_names.get(database_id, database_id),
                "indicators": count,
                "periodicities": "|".join(
                    f"{key}:{value}"
                    for key, value in db_periodicity[database_id].most_common()
                ),
                "coverage_min": min(coverage) if coverage else None,
                "coverage_median": statistics.median(coverage) if coverage else None,
                "coverage_max": max(coverage) if coverage else None,
                "span_start": db_span_start[database_id].isoformat()
                if database_id in db_span_start
                else "",
                "span_end": db_span_end[database_id].isoformat()
                if database_id in db_span_end
                else "",
                "top_l1_topics": " | ".join(
                    f"{name}:{value}" for name, value in top_l1
                ),
                "regions": "|".join(sorted(db_regions[database_id])),
            }
        )
    return pd.DataFrame(rows)


databases = build_databases(records)

DATASET_NAMES = fetch_dataset_names()
databases["database_label"] = [
    DATASET_NAMES.get(database_id) or database_name or database_id
    for database_id, database_name in zip(
        databases["database_id"], databases["database_name"]
    )
]
DB_LABELS = databases.set_index("database_id")["database_label"].to_dict()

print(f"Bases: {len(databases):,} · nombres: {len(DATASET_NAMES):,}")


Bases: 171 · nombres: 176


## 3. Cobertura regional

Se cruza `ref_country` con la jerarquía oficial país → región (`/portal/v1/hierarchy?type=country`). Para cada base y región se calcula la proporción media de países de la región cubiertos por sus indicadores. Se excluyen agregados (`WLD`, `EAS`, etc.).


In [5]:
def build_region_coverage(records: list[dict]) -> pd.DataFrame:
    regions = fetch_hierarchy()
    country_region: dict[str, set[str]] = defaultdict(set)
    region_size: dict[str, int] = {}
    region_name: dict[str, str] = {}

    for region in regions:
        region_id = region["id"]
        region_name[region_id] = region["name"]
        region_size[region_id] = len(region["countries"])
        for country in region["countries"]:
            country_region[country["id"]].add(region_id)

    db_region: dict[tuple[str, str], list[float]] = defaultdict(list)
    for record in records:
        database_id = record["database_id"] or "UNKNOWN"
        codes = {
            country.get("code")
            for country in record.get("ref_country") or []
            if country.get("code") and country.get("code") not in NON_COUNTRY
        }
        for region_id, size in region_size.items():
            if size == 0:
                continue
            covered = sum(
                1
                for country, region_ids in country_region.items()
                if region_id in region_ids and country in codes
            )
            db_region[(database_id, region_id)].append(covered / size)

    rows = []
    for (database_id, region_id), values in sorted(db_region.items()):
        rows.append(
            {
                "database_id": database_id,
                "region": region_id,
                "region_name": region_name[region_id],
                "region_countries": region_size[region_id],
                "indicators": len(values),
                "mean_coverage": round(sum(values) / len(values), 3),
                "median_coverage": round(sorted(values)[len(values) // 2], 3),
                "full_coverage_indicators": sum(1 for value in values if value >= 0.99),
            }
        )
    return pd.DataFrame(rows)


region_cov = build_region_coverage(records)
print(f"Cobertura regional: {region_cov.shape[0]:,} celdas base × región")


Cobertura regional: 1,539 celdas base × región


## 4. Frescura real

`time_periods.end` es nominal e incluye proyecciones. Para verificar el último dato real se consulta `/data360/data` con:

- todos los indicadores subanuales (mensuales / trimestrales);
- hasta 2 indicadores anuales por base, priorizando los de mayor cobertura.

Cada sonda prueba hasta 3 países y se queda con la observación más reciente. Es una **cota inferior**: no recorre todos los países de cada indicador.


In [6]:
def build_freshness(
    records: list[dict], indicators: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    by_id = {record["idno"]: record for record in records if record.get("idno")}
    jobs = []

    granular = indicators[indicators["periodicity"].isin(GRANULAR)]
    for _, row in granular.iterrows():
        meta = by_id.get(row["idno"], {})
        countries = [
            country["code"]
            for country in meta.get("ref_country") or []
            if country.get("code") and country["code"] not in NON_COUNTRY
        ]
        jobs.append(("granular", row, countries[:3]))

    annual = indicators[indicators["periodicity"] == "Annual"].copy()
    annual["_coverage"] = pd.to_numeric(
        annual["coverage_economies"], errors="coerce"
    ).fillna(0)
    for _, group in annual.groupby("database_id"):
        for _, row in group.sort_values("_coverage", ascending=False).head(2).iterrows():
            meta = by_id.get(row["idno"], {})
            countries = [
                country["code"]
                for country in meta.get("ref_country") or []
                if country.get("code") and country["code"] not in NON_COUNTRY
            ]
            jobs.append(("annual", row, countries[:3]))

    results: dict[tuple[str, str, str], tuple] = {}

    def probe(job):
        kind, row, countries = job
        for country in countries:
            latest, raw, count = latest_for(
                row["database_id"], row["idno"], country
            )
            if latest:
                return kind, row, country, latest, raw, count
        return kind, row, None, None, None, 0

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(probe, job) for job in jobs]
        for index, future in enumerate(as_completed(futures), 1):
            kind, row, country, latest, raw, count = future.result()
            results[(kind, row["database_id"], row["idno"])] = (
                country,
                latest,
                raw,
                count,
            )
            if index % 100 == 0:
                print(f"  sondas {index}/{len(jobs)}", flush=True)

    granular_rows = []
    for _, row in granular.iterrows():
        country, latest, raw, count = results.get(
            ("granular", row["database_id"], row["idno"]),
            (None, None, None, 0),
        )
        granular_rows.append(
            {
                "database_id": row["database_id"],
                "idno": row["idno"],
                "name": row["name"],
                "periodicity": row["periodicity"],
                "nominal_end": row["period_end"],
                "actual_latest": raw or "",
                "actual_latest_date": latest.isoformat() if latest else "",
                "probe_country": country or "",
                "rows_returned": count,
            }
        )

    actual_by_db: dict[str, list[dict]] = defaultdict(list)
    for _, row in annual.iterrows():
        country, latest, raw, count = results.get(
            ("annual", row["database_id"], row["idno"]),
            (None, None, None, 0),
        )
        if latest:
            actual_by_db[row["database_id"]].append(
                {
                    "idno": row["idno"],
                    "country": country,
                    "latest": raw,
                    "latest_date": latest,
                    "nominal_end": row["period_end"],
                }
            )

    db_rows = []
    for database_id, values in sorted(actual_by_db.items()):
        values.sort(key=lambda value: value["latest_date"], reverse=True)
        best = values[0]
        db_rows.append(
            {
                "database_id": database_id,
                "probe_indicator": best["idno"],
                "probe_country": best["country"],
                "actual_latest_period": best["latest"],
                "actual_latest_year": best["latest_date"].year,
                "nominal_end": best["nominal_end"],
                "is_projection_year": int(str(best["nominal_end"])[:4]) > REFERENCE_YEAR,
                "probes": len(values),
            }
        )

    return pd.DataFrame(granular_rows), pd.DataFrame(db_rows)


started = time.time()
granular_freshness, freshness = build_freshness(records, indicators)
print(
    f"Bases sondeadas: {len(freshness):,} · subanuales: {len(granular_freshness):,} · "
    f"{time.time() - started:.1f} s"
)


  sondas 100/338


  sondas 200/338


  sondas 300/338


Bases sondeadas: 159 · subanuales: 36 · 25.3 s


## 5. Visualizaciones

Las seis figuras usan las tablas construidas arriba. Los tooltips son interactivos y los gráficos se pueden exportar a HTML en una etapa posterior.


### 5.1 Treemap jerárquico

**Pregunta:** ¿cómo se distribuye el catálogo entre temas y subtemas?

**Lectura:** Prosperity es el bloque más grande y **Sin clasificar** el segundo: una porción importante del catálogo no tiene tema L0 asignado. Luego aparecen People, Infrastructure, Planet y Digital, todos mucho más chicos. El treemap muestra que el catálogo está concentrado en pocos temas y que la clasificación temática tiene un hueco relevante.


In [7]:
def primary_topic(series: pd.Series) -> pd.Series:
    return series.fillna("").astype(str).str.split("|").str[0].str.strip()


indicators["l0"] = primary_topic(indicators["topic_l0"]).replace(
    "", "Sin clasificar"
)
indicators["l1"] = primary_topic(indicators["topic_l1"]).replace(
    "", "Sin subclasificar"
)
indicators["l2"] = primary_topic(indicators["topic_l2"]).replace("", "Sin detalle")

W, H = 100.0, 60.0
LABEL_MIN_AREA = 60.0
LABEL_MIN_WIDTH = 5.0
LABEL_MIN_HEIGHT = 3.0
LABEL_PX_PER_UNIT = 9.0
LABEL_CHAR_PX = 4.2
L0_LABEL_INSET = 0.1
L0_LABEL_PAD_X = 0.35
L0_LABEL_PAD_Y = 0.2
L0_LABEL_BG_HEIGHT = 2.0
L0_LABEL_CHAR_W = 0.65


def short_label(value: str, width: int) -> str:
    return value if len(value) <= width else value[: width - 1] + "…"


def fit_label(value: str, block_width: float) -> str:
    max_chars = int((block_width * LABEL_PX_PER_UNIT - 4) / LABEL_CHAR_PX)
    return short_label(value, width=max_chars) if max_chars >= 8 else ""


def block_label(row) -> str:
    if (
        row["area"] < LABEL_MIN_AREA
        or row["width"] < LABEL_MIN_WIDTH
        or row["height"] < LABEL_MIN_HEIGHT
    ):
        return ""
    name = fit_label(row["l2"], row["width"])
    return f"{name}\n{row['indicators']:,}" if name else ""


def l0_bg_width(label: str) -> float:
    return len(label) * L0_LABEL_CHAR_W + 2 * L0_LABEL_PAD_X


def inset(rect, fraction):
    return (
        rect["x"] + rect["dx"] * fraction,
        rect["y"] + rect["dy"] * fraction,
        rect["dx"] * (1 - 2 * fraction),
        rect["dy"] * (1 - 2 * fraction),
    )


def treemap_layout(sizes, x, y, w, h):
    normalized = squarify.normalize_sizes(list(sizes), w, h)
    return squarify.squarify(normalized, x, y, w, h)


leaves = []
l1_boxes = []
l0_boxes = []
l0_labels = []

l0_counts = indicators.groupby("l0").size().sort_values(ascending=False)
L0_ORDER = l0_counts.index.tolist()
L0_COLORS = [
    THEME_COLORS.get(name, CATEGORICAL_PALETTE[index % len(CATEGORICAL_PALETTE)])
    for index, name in enumerate(L0_ORDER)
]

for (l0, _), r0 in zip(l0_counts.items(), treemap_layout(l0_counts.values, 0, 0, W, H)):
    l0_boxes.append(
        {"x": r0["x"], "y": r0["y"], "x2": r0["x"] + r0["dx"], "y2": r0["y"] + r0["dy"]}
    )
    bg_x = r0["x"] + L0_LABEL_INSET
    bg_y2 = r0["y"] + r0["dy"] - L0_LABEL_INSET
    l0_labels.append(
        {
            "l0": l0,
            "text_x": bg_x + L0_LABEL_PAD_X,
            "text_y": bg_y2 - L0_LABEL_PAD_Y,
            "bg_x": bg_x,
            "bg_x2": bg_x + l0_bg_width(l0),
            "bg_y": bg_y2 - L0_LABEL_BG_HEIGHT,
            "bg_y2": bg_y2,
        }
    )
    x0, y0, w0, h0 = inset(r0, 0.010)
    sub0 = indicators[indicators["l0"] == l0]
    l1_counts = sub0.groupby("l1").size().sort_values(ascending=False)
    for (l1, _), r1 in zip(
        l1_counts.items(), treemap_layout(l1_counts.values, x0, y0, w0, h0)
    ):
        l1_boxes.append(
            {
                "x": r1["x"],
                "y": r1["y"],
                "x2": r1["x"] + r1["dx"],
                "y2": r1["y"] + r1["dy"],
            }
        )
        x1, y1, w1, h1 = inset(r1, 0.015)
        sub1 = sub0[sub0["l1"] == l1]
        l2_counts = sub1.groupby("l2").size().sort_values(ascending=False)
        for (l2, _), r2 in zip(
            l2_counts.items(), treemap_layout(l2_counts.values, x1, y1, w1, h1)
        ):
            leaves.append(
                {
                    "l0": l0,
                    "l1": l1,
                    "l2": l2,
                    "indicators": int(l2_counts[l2]),
                    "x": r2["x"],
                    "y": r2["y"],
                    "x2": r2["x"] + r2["dx"],
                    "y2": r2["y"] + r2["dy"],
                    "width": r2["dx"],
                    "height": r2["dy"],
                }
            )

treemap_df = pd.DataFrame(leaves)
treemap_df["cx"] = (treemap_df["x"] + treemap_df["x2"]) / 2
treemap_df["cy"] = (treemap_df["y"] + treemap_df["y2"]) / 2
treemap_df["area"] = treemap_df["width"] * treemap_df["height"]
treemap_df["label"] = treemap_df.apply(block_label, axis=1)
l1_df = pd.DataFrame(l1_boxes)
l0_df = pd.DataFrame(l0_boxes)
l0_label_df = pd.DataFrame(l0_labels)

print(f"Treemap: {len(treemap_df)} hojas L2 · {treemap_df['indicators'].sum():,} indicadores")

treemap = (
    alt.Chart(treemap_df)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W]), axis=None),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H]), axis=None),
        y2="y2:Q",
        color=alt.Color(
            "l0:N",
            title="Tema L0",
            scale=alt.Scale(domain=L0_ORDER, range=L0_COLORS),
        ),
        tooltip=[
            alt.Tooltip("l0:N", title="Tema L0"),
            alt.Tooltip("l1:N", title="Tema L1"),
            alt.Tooltip("l2:N", title="Tema L2"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
        ],
    )
)

l1_borders = (
    alt.Chart(l1_df)
    .mark_rect(fill=None, stroke="white", strokeWidth=1)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W])),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H])),
        y2="y2:Q",
    )
)

l0_borders = (
    alt.Chart(l0_df)
    .mark_rect(fill=None, stroke="#1b1b1b", strokeWidth=2)
    .encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=[0, W])),
        x2="x2:Q",
        y=alt.Y("y:Q", scale=alt.Scale(domain=[0, H])),
        y2="y2:Q",
    )
)

leaf_labels = (
    alt.Chart(treemap_df[treemap_df["label"] != ""])
    .mark_text(
        fontSize=7,
        color="white",
        lineBreak="\n",
        limit=300,
        align="center",
        baseline="middle",
    )
    .encode(
        x=alt.X("cx:Q", scale=alt.Scale(domain=[0, W])),
        y=alt.Y("cy:Q", scale=alt.Scale(domain=[0, H])),
        text=alt.Text("label:N"),
    )
)

l0_label_bg = (
    alt.Chart(l0_label_df)
    .mark_rect(
        fill="white",
        opacity=0.92,
        stroke="#1b1b1b",
        strokeWidth=0.5,
        cornerRadius=2,
    )
    .encode(
        x=alt.X("bg_x:Q", scale=alt.Scale(domain=[0, W])),
        x2="bg_x2:Q",
        y=alt.Y("bg_y:Q", scale=alt.Scale(domain=[0, H])),
        y2="bg_y2:Q",
    )
)

l0_text = (
    alt.Chart(l0_label_df)
    .mark_text(
        align="left", baseline="top", fontSize=12, fontWeight="bold", color="#1b1b1b"
    )
    .encode(
        x=alt.X("text_x:Q", scale=alt.Scale(domain=[0, W])),
        y=alt.Y("text_y:Q", scale=alt.Scale(domain=[0, H])),
        text=alt.Text("l0:N"),
    )
)

(
    treemap + l1_borders + l0_borders + leaf_labels + l0_label_bg + l0_text
).properties(
    width=900,
    height=540,
    title=alt.TitleParams(
        text="Estructura temática del catálogo",
        subtitle="Área = cantidad de indicadores · borde negro = tema L0 · borde blanco = subtema",
    ),
)


Treemap: 190 hojas L2 · 9,859 indicadores


alt.LayerChart(...)

### 5.2 Top-N bases por cantidad de indicadores

**Pregunta:** ¿qué bases concentran el catálogo?  
**Codificación:** barras horizontales, `x = indicadores`, `y = base`, `color = tema L1 dominante`.  
**Fuente:** metadatos de la API agrupados por `database_id`.

**Lectura:** el catálogo está muy concentrado: unas pocas bases explican la mayor parte de los indicadores.


In [8]:
def dominant_l1(value: str) -> str:
    if not isinstance(value, str) or not value.strip():
        return "Sin clasificar"
    return value.split("|")[0].split(":")[0].strip()


databases["dominant_l1"] = databases["top_l1_topics"].map(dominant_l1)
top20 = databases.sort_values("indicators", ascending=False).head(20).copy()

share = top20["indicators"].sum() / databases["indicators"].sum()
print(f"Top 20: {top20['indicators'].sum():,} indicadores ({share:.1%} del catálogo)")

bars = (
    alt.Chart(top20)
    .mark_bar(cornerRadiusEnd=3)
    .encode(
        x=alt.X("indicators:Q", title="Cantidad de indicadores"),
        y=alt.Y(
            "database_label:N",
            sort="-x",
            title=None,
            axis=alt.Axis(labelLimit=260),
        ),
        color=alt.Color(
            "dominant_l1:N",
            title="Tema L1 dominante",
            scale=alt.Scale(range=CATEGORICAL_PALETTE),
        ),
        tooltip=[
            alt.Tooltip("database_label:N", title="Base"),
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("dominant_l1:N", title="Tema L1"),
        ],
    )
    .properties(width=760, height=460, title="Top 20 bases por cantidad de indicadores")
)

labels = bars.mark_text(align="left", dx=4, fontSize=10).encode(
    text=alt.Text("indicators:Q", format=","), color=alt.value("#333333")
)

(bars + labels).configure_axis(grid=True, gridOpacity=0.2)


Top 20: 7,447 indicadores (75.5% del catálogo)


alt.LayerChart(...)

### 5.3 Heatmap base × región

**Pregunta:** ¿qué bases tienen vacíos regionales?  
**Codificación:** color = cobertura media (0–1) de los indicadores de la base en cada región.  
**Fuente:** `ref_country` cruzado con la jerarquía país → región.

Para que sea legible se muestran las **25 bases con más indicadores** más las 8 del shortlist temático. El color se agrupa en **5 intervalos discretos**.

**Lectura:** las bases financieras y de gobernanza muestran los vacíos regionales más marcados.


In [9]:
SHORTLIST = {
    "PEERING_DB": "Datacenters",
    "PCH_IXP": "IXPs",
    "OOKLA_ST": "Velocidad",
    "ITU_DH": "Conectividad",
    "WB_WDI": "Internet / energía",
    "WB_SE4ALL": "Energía",
    "WB_RISE": "Energía (política)",
    "WB_PPI": "Inversión infra",
}

REGION_ORDER = ["AFE", "AFW", "EAS", "ECS", "LCN", "MEA", "NAC", "SAS", "SSF"]

heatmap_ids = list(
    dict.fromkeys(
        databases.sort_values("indicators", ascending=False)
        .head(25)["database_id"]
        .tolist()
        + list(SHORTLIST)
    )
)
heatmap_df = region_cov[region_cov["database_id"].isin(heatmap_ids)].copy()
heatmap_df["shortlist"] = heatmap_df["database_id"].isin(SHORTLIST)
heatmap_df["database_label"] = heatmap_df["database_id"].map(DB_LABELS)

REGION_LABELS = {
    "AFE": "África Oriental y Austral",
    "AFW": "África Occidental y Central",
    "EAS": "Asia Oriental y Pacífico",
    "ECS": "Europa y Asia Central",
    "LCN": "América Latina y Caribe",
    "MEA": "Medio Oriente y Pakistán",
    "NAC": "Norteamérica",
    "SAS": "Asia del Sur",
    "SSF": "África Subsahariana",
}
heatmap_df["region_label"] = heatmap_df["region"].map(REGION_LABELS)
REGION_LABEL_ORDER = [REGION_LABELS[code] for code in REGION_ORDER]
heatmap_labels = [DB_LABELS.get(database_id, database_id) for database_id in heatmap_ids]

COVERAGE_LABELS = ["0–20 %", "20–40 %", "40–60 %", "60–80 %", "80–100 %"]
heatmap_df["coverage_band"] = pd.cut(
    heatmap_df["mean_coverage"],
    bins=[-0.001, 0.2, 0.4, 0.6, 0.8, 1.0],
    labels=COVERAGE_LABELS,
)

print(f"Heatmap: {len(heatmap_ids)} bases × {len(REGION_ORDER)} regiones")

heat = (
    alt.Chart(heatmap_df)
    .mark_rect(stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "region_label:N",
            sort=REGION_LABEL_ORDER,
            title=None,
            axis=alt.Axis(labelAngle=-45, labelLimit=120),
        ),
        y=alt.Y(
            "database_label:N",
            sort=heatmap_labels,
            title=None,
            axis=alt.Axis(labelLimit=240),
        ),
        color=alt.Color(
            "coverage_band:N",
            title="Cobertura media",
            sort=COVERAGE_LABELS,
            scale=alt.Scale(
                domain=COVERAGE_LABELS,
                range=COVERAGE_COLORS,
                type="ordinal",
            ),
        ),
        tooltip=[
            alt.Tooltip("database_label:N", title="Base"),
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("region_label:N", title="Región"),
            alt.Tooltip("region:N", title="Código región"),
            alt.Tooltip("coverage_band:N", title="Intervalo"),
            alt.Tooltip("mean_coverage:Q", title="Cobertura media", format=".3f"),
            alt.Tooltip("median_coverage:Q", title="Cobertura mediana", format=".3f"),
            alt.Tooltip("indicators:Q", title="Indicadores"),
            alt.Tooltip("region_countries:Q", title="Países en la región"),
        ],
    )
)

heat_text = heat.mark_text(fontSize=7).encode(
    text=alt.Text("mean_coverage:Q", format=".0%"),
    color=alt.condition(
        alt.datum.mean_coverage > 0.6, alt.value("white"), alt.value("#333333")
    ),
)

(heat + heat_text).properties(
    width=760,
    height=620,
    title="Cobertura media por base × región (top 25 + shortlist)",
)


Heatmap: 32 bases × 9 regiones


alt.LayerChart(...)

### 5.4 Dumbbell: fin nominal vs. último dato real

**Pregunta:** ¿la metadata promete datos más recientes de los que realmente existen?  
**Codificación:** un par de puntos por base (fin nominal y último dato observado), unidos por una línea; se muestran las 30 bases con mayor brecha absoluta.  
**Fuente:** `time_periods.end` (metadata) vs. `/data360/data` (observaciones reales).

- `gap = nominal_end − actual_latest_year`.
- `gap > 0` → la metadata declara un fin posterior al último dato real (desactualización / proyección).
- `gap < 0` → hay datos más allá del fin declarado (metadata conservadora).

**Lectura:** varias bases declaran un fin nominal posterior al último dato real; la brecha es más visible en series de proyección o desactualizadas.


In [10]:
freshness = freshness.copy()
freshness["actual_year"] = pd.to_numeric(
    freshness["actual_latest_period"].astype(str).str.extract(r"(\d{4})")[0],
    errors="coerce",
)
freshness["nominal_year"] = pd.to_numeric(
    freshness["nominal_end"].astype(str).str.extract(r"(\d{4})")[0],
    errors="coerce",
)
freshness = freshness.dropna(subset=["actual_year", "nominal_year"]).copy()
freshness[["actual_year", "nominal_year"]] = freshness[
    ["actual_year", "nominal_year"]
].astype(int)
freshness["gap"] = freshness["nominal_year"] - freshness["actual_year"]
freshness["estado"] = freshness["gap"].map(
    lambda gap: "Metadata sobreestima"
    if gap > 0
    else ("Metadata subestima" if gap < 0 else "Coincide")
)

counts = freshness["estado"].value_counts()
print(
    f"Frescura: {len(freshness)} bases · "
    f"{counts.get('Metadata sobreestima', 0)} sobreestiman · "
    f"{counts.get('Metadata subestima', 0)} subestiman · "
    f"{counts.get('Coincide', 0)} coinciden"
)

shown = freshness.reindex(
    freshness["gap"].abs().sort_values(ascending=False).index
).head(30).copy()
shown["database_label"] = shown["database_id"].map(DB_LABELS)

db_l0 = (
    indicators.groupby(["database_id", "l0"]).size()
    .reset_index(name="n")
    .sort_values(["database_id", "n"], ascending=[True, False])
    .drop_duplicates("database_id")
    .set_index("database_id")["l0"]
)
shown["l0"] = shown["database_id"].map(db_l0).fillna("Sin clasificar")

l0_rank = {name: i for i, name in enumerate(THEME_COLORS)}
shown["l0_rank"] = shown["l0"].map(l0_rank).fillna(len(l0_rank))
shown = shown.sort_values(["l0_rank", "gap"], ascending=[True, False])
order = shown["database_label"].tolist()

long = shown.melt(
    id_vars=["database_id", "database_label", "l0", "gap", "estado"],
    value_vars=["actual_year", "nominal_year"],
    var_name="serie",
    value_name="year",
)
long["serie"] = long["serie"].map(
    {"actual_year": "Último dato real", "nominal_year": "Fin nominal (metadata)"}
)

rule = (
    alt.Chart(shown)
    .mark_rule(color=PERIODICITY_COLORS["Desconocida"], strokeWidth=2)
    .encode(
        y=alt.Y(
            "database_label:N",
            sort=order,
            title=None,
            axis=alt.Axis(labelLimit=260),
        ),
        x=alt.X("actual_year:Q", title="Año", scale=alt.Scale(zero=False)),
        x2="nominal_year:Q",
    )
)

points = (
    alt.Chart(long)
    .mark_circle(size=90, opacity=0.9)
    .encode(
        y=alt.Y(
            "database_label:N",
            sort=order,
            title=None,
            axis=alt.Axis(labelLimit=260),
        ),
        x=alt.X("year:Q", title="Año", scale=alt.Scale(zero=False)),
        color=alt.Color(
            "serie:N",
            title=None,
            scale=alt.Scale(
                domain=["Último dato real", "Fin nominal (metadata)"],
                range=[
                    DUMBBELL_COLORS["Último dato real"],
                    DUMBBELL_COLORS["Fin nominal (metadata)"],
                ],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_label:N", title="Base"),
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("l0:N", title="Tema L0"),
            alt.Tooltip("serie:N", title="Serie"),
            alt.Tooltip("year:Q", title="Año"),
            alt.Tooltip("gap:Q", title="Brecha (años)"),
            alt.Tooltip("estado:N", title="Estado"),
        ],
    )
)

l0_domain = [name for name in THEME_COLORS if name in shown["l0"].unique()]
l0_range = [THEME_COLORS[name] for name in l0_domain]

group_band = (
    alt.Chart(shown)
    .mark_rect()
    .encode(
        y=alt.Y("database_label:N", sort=order, title=None, axis=None),
        color=alt.Color(
            "l0:N",
            title="Tema L0",
            scale=alt.Scale(domain=l0_domain, range=l0_range),
        ),
        tooltip=[alt.Tooltip("l0:N", title="Tema L0")],
    )
    .properties(width=12, height=640)
)

alt.hconcat(
    group_band,
    (rule + points).properties(width=760, height=640),
).resolve_scale(y="shared").properties(
    title="Fin nominal vs. último dato real — 30 mayores brechas"
)


Frescura: 159 bases · 40 sobreestiman · 4 subestiman · 115 coinciden


alt.HConcatChart(...)

### 5.5 Barras apiladas de periodicidad

**Pregunta:** ¿qué bases del catálogo no son puramente anuales?  
**Codificación:** composición porcentual de cada base; se muestran **solo las bases que no son 100 % anuales**. Las restantes se omiten porque una barra completamente anual no aporta información.  
**Fuente:** metadatos de la API (`periodicity`).

Cada barra se etiqueta con el **nombre legible** de la base (el código queda en el tooltip). Las periodicidades se muestran sin agrupar y traducidas al español. El color es único por tipo de periodicidad y se repite en todas las barras.

**Lectura:** la periodicidad anual es casi universal; las excepciones son bases específicas (encuestas, precios, índices).


In [11]:
PERIODICITY_LABELS = {
    "Annual": "Anual",
    "Triennial": "Trienal",
    "Biennial": "Bienal",
    "Every two years": "Cada dos años",
    "Five-Year Interval": "Quinquenal",
    "Ad hoc [adhoc]": "Ad hoc (irregular)",
    "Monthly": "Mensual",
    "Quarterly": "Trimestral",
    "Other": "Otra",
    "Unknown": "Desconocida",
}

per_base = (
    indicators.groupby(["database_id", "periodicity"]).size().reset_index(name="count")
)
base_totals = per_base.groupby("database_id")["count"].sum()
shares = (
    per_base.pivot_table(
        index="database_id", columns="periodicity", values="count", fill_value=0
    )
    .div(base_totals, axis=0)
    .fillna(0)
)
shares["annual_share"] = shares.get("Annual", 0)
shares["quarterly_share"] = shares.get("Quarterly", 0)
shares["monthly_share"] = shares.get("Monthly", 0)
shares["other_share"] = 1 - shares[
    ["annual_share", "quarterly_share", "monthly_share"]
].sum(axis=1)
non_annual = shares[shares["annual_share"] < 1].copy()
non_annual["has_quarterly"] = (non_annual["quarterly_share"] > 0).astype(int)
non_annual["has_monthly"] = (non_annual["monthly_share"] > 0).astype(int)
non_annual = non_annual.sort_values(
    ["annual_share", "has_quarterly", "has_monthly", "other_share", "database_id"],
    ascending=[False, False, False, False, True],
)
non_annual_ids = non_annual.index.tolist()

per_non = per_base[per_base["database_id"].isin(non_annual_ids)].copy()
per_non["database_label"] = per_non["database_id"].map(
    lambda database_id: DB_LABELS.get(database_id, database_id)
)
per_non["periodicity_label"] = per_non["periodicity"].map(
    lambda value: PERIODICITY_LABELS.get(value, value)
)
per_non["database_name"] = per_non["database_label"]

label_order = [
    DB_LABELS.get(database_id, database_id) for database_id in non_annual_ids
]
PERIODICITY_DOMAIN = [
    "Anual",
    "Trimestral",
    "Mensual",
    "Trienal",
    "Bienal",
    "Cada dos años",
    "Quinquenal",
    "Ad hoc (irregular)",
    "Otra",
    "Desconocida",
]
present_labels = set(per_non["periodicity_label"])
periodicity_order = [label for label in PERIODICITY_DOMAIN if label in present_labels]
periodicity_palette = [PERIODICITY_COLORS[label] for label in periodicity_order]

print(f"Bases no 100 % anuales: {len(non_annual_ids)} de {len(base_totals)}")

chart = (
    alt.Chart(per_non)
    .mark_bar()
    .encode(
        y=alt.Y(
            "database_label:N",
            sort=label_order,
            title=None,
            axis=alt.Axis(labelLimit=260),
        ),
        x=alt.X("count:Q", stack="normalize", title="Proporción de indicadores"),
        color=alt.Color(
            "periodicity_label:N",
            title="Periodicidad",
            scale=alt.Scale(domain=periodicity_order, range=periodicity_palette),
        ),
        tooltip=[
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("database_name:N", title="Nombre"),
            alt.Tooltip("periodicity_label:N", title="Periodicidad"),
            alt.Tooltip("count:Q", title="Indicadores", format=","),
        ],
    )
    .properties(
        width=760,
        height=len(non_annual_ids) * 32 + 60,
        title=alt.TitleParams(
            text=f"Bases con periodicidad no 100 % anual ({len(non_annual_ids)} de {len(base_totals)})",
            subtitle=f"Se omiten las {len(base_totals) - len(non_annual_ids)} bases cuyos indicadores son todos anuales",
        ),
    )
)

chart


Bases no 100 % anuales: 13 de 171


alt.Chart(...)

### 5.6 Scatter amplitud vs. profundidad

**Pregunta:** ¿las bases con más indicadores también cubren más economías?  
**Codificación:** `x = indicadores` (escala log), `y = cobertura mediana`, `tamaño = cobertura máxima`, color = pertenencia al shortlist temático.  
**Fuente:** metadatos de la API agrupados por `database_id`.

El shortlist son las 8 bases del recorte datacenter / conectividad / energía / infraestructura (`PEERING_DB`, `PCH_IXP`, `OOKLA_ST`, `ITU_DH`, `WB_WDI`, `WB_SE4ALL`, `WB_RISE`, `WB_PPI`).

**Lectura:** las bases del shortlist no son las más grandes, pero tienen coberturas altas; el tamaño no garantiza profundidad.


In [12]:
databases["grupo"] = databases["database_id"].map(
    lambda database_id: "Shortlist temático"
    if database_id in SHORTLIST
    else "Resto del catálogo"
)
databases["shortlist_role"] = databases["database_id"].map(SHORTLIST).fillna("")
shortlist_indicators = databases.loc[
    databases["database_id"].isin(SHORTLIST), "indicators"
].sum()
print(f"Shortlist: {len(SHORTLIST)} bases · {shortlist_indicators:,} indicadores")

base = (
    alt.Chart(databases)
    .mark_circle(opacity=0.65, stroke="white", strokeWidth=0.5)
    .encode(
        x=alt.X(
            "indicators:Q",
            title="Cantidad de indicadores (escala log)",
            scale=alt.Scale(type="log"),
        ),
        y=alt.Y("coverage_median:Q", title="Cobertura mediana (economías)"),
        size=alt.Size(
            "coverage_max:Q",
            title="Cobertura máxima",
            scale=alt.Scale(range=[20, 450]),
        ),
        color=alt.Color(
            "grupo:N",
            title=None,
            scale=alt.Scale(
                domain=["Shortlist temático", "Resto del catálogo"],
                range=[
                    GROUP_COLORS["Shortlist temático"],
                    GROUP_COLORS["Resto del catálogo"],
                ],
            ),
        ),
        tooltip=[
            alt.Tooltip("database_label:N", title="Base"),
            alt.Tooltip("database_id:N", title="Código"),
            alt.Tooltip("shortlist_role:N", title="Rol en el shortlist"),
            alt.Tooltip("indicators:Q", title="Indicadores", format=","),
            alt.Tooltip("coverage_median:Q", title="Cobertura mediana"),
            alt.Tooltip("coverage_max:Q", title="Cobertura máxima"),
        ],
    )
)

labels = (
    alt.Chart(databases[databases["database_id"].isin(SHORTLIST)])
    .mark_text(dx=7, dy=-5, fontSize=10, fontWeight="bold")
    .encode(
        x="indicators:Q",
        y="coverage_median:Q",
        text="database_id:N",
        color=alt.value(GROUP_TEXT_COLOR),
    )
)

(base + labels).properties(
    width=660, height=440, title="Amplitud vs. profundidad del catálogo (shortlist resaltado)"
).interactive()


Shortlist: 8 bases · 1,629 indicadores


alt.LayerChart(...)

## Hallazgos y próximos pasos

- El catálogo está concentrado: el top 20 de bases explica cerca de tres cuartos de los indicadores.
- La frescura nominal no es confiable: una parte relevante de las bases declara un fin posterior al último dato real.
- La periodicidad anual es casi universal; las series subanuales son una excepción.
- El shortlist temático combina pocas bases con coberturas altas: es un buen punto de partida para las visualizaciones periodísticas.
- Las cifras exactas de la corrida se imprimen en la celda siguiente.
- Para publicar, los specs de Altair se pueden exportar a HTML o reutilizar en Vega-Lite.


In [13]:
top20_share = (
    databases.sort_values("indicators", ascending=False).head(20)["indicators"].sum()
    / databases["indicators"].sum()
)
annual_share = (indicators["periodicity"] == "Annual").mean()
over = int((freshness["gap"] > 0).sum())
under = int((freshness["gap"] < 0).sum())

print(f"Fecha de descarga: {REFERENCE_DATE.isoformat()}")
print(f"Bases: {len(databases):,} · indicadores: {len(indicators):,}")
print(f"Top 20 bases: {top20_share:.1%} de los indicadores")
print(f"Indicadores anuales: {annual_share:.1%}")
print(
    f"Frescura: {over} bases sobreestiman, {under} subestiman, "
    f"{len(freshness) - over - under} coinciden"
)


Fecha de descarga: 2026-11-27
Bases: 171 · indicadores: 9,859
Top 20 bases: 75.5% de los indicadores
Indicadores anuales: 95.3%
Frescura: 40 bases sobreestiman, 4 subestiman, 115 coinciden
